# 00 · Dati, split fisso e scelta di `max_seq_length`

Da eseguire **una volta** sul dataset completo. Produce il file di split che baseline, fine-tuning e valutazione finale rileggono: se cambia lo split, i numeri non sono più confrontabili.

1. Caricamento parquet di segmenti e filtro sul perimetro macro → micro (`slm_baseline/labels.py`)
2. Report su righe fuori perimetro e testo vuoto
3. Split train / val / test stratificato per `macro::micro` e raggruppato per `document_id`
4. Lunghezze in token con i tokenizer dei tre modelli → scelta di `max_seq_length`

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))  # works even without `pip install -e .`

## Config

**Due modalità:** `CONFIG_PATH` → `configs/experiments/*.yaml` (come il CLI) oppure `USE_EXPERIMENT_YAML = False` e path legacy.

Con `USE_EXPERIMENT_YAML` e `USE_CLI_PREPARE=True` (default), la cella sotto esegue **Step 1** (`cli prepare`). Le sezioni EDA riusano quel split; `make_splits` manuale si salta.

In [ ]:
USE_EXPERIMENT_YAML = True
USE_CLI_PREPARE = True  # Step 1: stesso comando di `python -m slm_baseline.cli prepare`
FORCE_PREPARE = False   # True → --force se cambi YAML su esperimento esistente
CONFIG_PATH = ROOT / "configs" / "experiments" / "holdout_s42_minimal.yaml"

# Legacy se USE_EXPERIMENT_YAML = False
DATA_PATH = ROOT / "data" / "sample" / "df_sample_150.parquet"
SPLITS_PATH = ROOT / "data" / "splits_v1.parquet"
TEXT_COLUMN = "ocr_surrogates"

TEST_FRAC = 0.07
VAL_FRAC = 0.05
SEED = 42
OVERWRITE_SPLITS = False

TOKENIZERS = ["Qwen/Qwen3-1.7B", "HuggingFaceTB/SmolLM3-3B", "google/gemma-3-4b-it"]
TOKEN_SAMPLE = None
HEADROOM = 0.20
RUN_CONFIG_PATH = ROOT / "data" / "run_config.json"
PROMPT_VERSION = "minimal"

if USE_EXPERIMENT_YAML:
    import os
    from slm_baseline.experiment import load_experiment_config, load_merged_experiment_dict, resolved_dataset_path

    os.environ.setdefault("TESI_REPO_ROOT", str(ROOT))
    os.environ.setdefault("TESI_DATA_DIR", str(ROOT))
    os.environ.setdefault("TESI_OUTPUT_DIR", str(ROOT / "outputs"))
    _cfg = load_experiment_config(CONFIG_PATH)
    _merged = load_merged_experiment_dict(CONFIG_PATH)
    DATA_PATH = resolved_dataset_path(_cfg)
    EXP_DIR = ROOT / "outputs" / "experiments" / _cfg.experiment_id
    SPLITS_PATH = EXP_DIR / "splits.parquet"
    RUN_CONFIG_PATH = EXP_DIR / "run_config.json"
    TEXT_COLUMN = _cfg.dataset.text_column
    TEST_FRAC = _cfg.split.test_frac
    VAL_FRAC = _cfg.split.val_frac
    SEED = _cfg.split.seed
    TOKENIZERS = list(_cfg.sequence.tokenizers)
    HEADROOM = _cfg.sequence.headroom
    PROMPT_VERSION = _cfg.prompt.version

    if USE_CLI_PREPARE:
        import subprocess

        cmd = [sys.executable, "-m", "slm_baseline.cli", "prepare", "--config", str(CONFIG_PATH)]
        if FORCE_PREPARE:
            cmd.append("--force")
        env = {**os.environ, "PYTHONPATH": str(ROOT / "src")}
        proc = subprocess.run(cmd, cwd=ROOT, env=env)
        if proc.returncode != 0:
            raise RuntimeError("cli prepare fallito")
        print(f"Step 1 prepare OK → {EXP_DIR}")

## 1. Caricamento e perimetro

In [ ]:
import pandas as pd
from slm_baseline.data import (
    attach_splits, check_no_group_leakage, filter_perimeter, label_distribution,
    load_segments, make_splits, prepare_text,
)

raw = load_segments(DATA_PATH)
segments, out_of_perimeter = filter_perimeter(raw)
print(f"righe totali: {len(raw)}  in perimetro: {len(segments)}  fuori: {len(out_of_perimeter)}")
if len(out_of_perimeter):
    display(out_of_perimeter.groupby(["macro_category", "sub_category"]).size().rename("n").to_frame())

In [ ]:
# Righe senza testo per ciascuna colonna candidata: vengono escluse in valutazione, non dallo split
for col in ["ocr_surrogates", "ocr_anonymized"]:
    if col in segments:
        _, empty = prepare_text(segments, col)
        print(f"{col}: {len(empty)} righe con testo vuoto")

In [ ]:
dist = segments.groupby(["macro_category", "sub_category"]).size().rename("n").to_frame()
dist["pct"] = (dist["n"] / dist["n"].sum() * 100).round(2)
dist

## 2. Split fisso

In [ ]:
if USE_EXPERIMENT_YAML and USE_CLI_PREPARE:
    splits = pd.read_parquet(SPLITS_PATH)
    print(f"split da cli prepare: {SPLITS_PATH}")
elif SPLITS_PATH.exists() and not OVERWRITE_SPLITS:
    splits = pd.read_parquet(SPLITS_PATH)
    print(f"split esistente riusato: {SPLITS_PATH}")
    unknown = set(segments["record_key"]) - set(splits["record_key"])
    if unknown:
        raise ValueError(
            f"{len(unknown)} record non presenti nello split esistente: dataset diverso da quello dello split. "
            "Usa un nuovo SPLITS_PATH (es. splits_v2) oppure OVERWRITE_SPLITS=True se nessun risultato dipende ancora da questo split."
        )
else:
    splits = make_splits(segments, test_frac=TEST_FRAC, val_frac=VAL_FRAC, seed=SEED)
    SPLITS_PATH.parent.mkdir(parents=True, exist_ok=True)
    splits.to_parquet(SPLITS_PATH, index=False)
    print(f"split salvato: {SPLITS_PATH}")

check_no_group_leakage(splits)
print(splits["split"].value_counts().to_dict())
label_distribution(splits)

## 3. Lunghezze in token → `max_seq_length`

Si misura la sequenza **completa di training** (system + macro + testo + risposta) con il chat template di ciascun modello. La regola: scegliere il valore che copre il 95–97% degli esempi; sopra la soglia si tronca solo il testo del segmento, mai istruzioni o risposta.

Su SageMaker serve `HF_TOKEN` per il tokenizer di Gemma (licenza da accettare sulla pagina del modello).

In [ ]:
from transformers import AutoTokenizer
from slm_baseline.inference import token_length_report

usable, _ = prepare_text(segments, TEXT_COLUMN)
# Tutti gli split: max_seq_length deve coprire anche il segmento più lungo di val/test
all_rows = attach_splits(usable, splits)
sample = all_rows if TOKEN_SAMPLE is None else all_rows.sample(n=min(TOKEN_SAMPLE, len(all_rows)), random_state=SEED)

lengths, coverage = {}, {}
for name in TOKENIZERS:
    tok = AutoTokenizer.from_pretrained(name)
    lengths[name], coverage[name] = token_length_report(tok, sample, prompt_version=PROMPT_VERSION)

pd.concat({n.split("/")[-1]: c.set_index("max_seq_len")["pct_covered"] for n, c in coverage.items()}, axis=1)

In [ ]:
pd.DataFrame({n.split("/")[-1]: l.describe(percentiles=[0.5, 0.9, 0.95, 0.99]) for n, l in lengths.items()}).round(0)

In [ ]:
# Copertura per categoria: decidere sulla categoria peggiore, non sulla media
THRESHOLDS = (4096, 8192, 16384)
ref = lengths[TOKENIZERS[0]]
per_cat = sample.assign(seq_tokens=ref).groupby(["macro_category", "sub_category"])["seq_tokens"]
table = pd.DataFrame({"n": per_cat.size(), "p50": per_cat.median(), "p95": per_cat.quantile(0.95).round(), "max": per_cat.max()})
for t in THRESHOLDS:
    table[f"% > {t}"] = per_cat.apply(lambda s, t=t: round(float((s > t).mean()) * 100, 1))
table.sort_values(f"% > {THRESHOLDS[0]}", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for n, l in lengths.items():
    ax.hist(l.clip(upper=8192), bins=60, histtype="step", label=n.split("/")[-1])
for t in (2048, 4096):
    ax.axvline(t, ls="--", c="grey")
ax.set_xlabel("token per esempio (clip a 8192)"); ax.set_ylabel("esempi"); ax.legend()
plt.show()

### Scelta di `max_seq_length`

Obiettivo: coprire il 100% dei segmenti con un margine del 20% sopra il più lungo (misurato con tutti e tre i tokenizer), arrotondato al multiplo di 1024. Il valore viene scritto in `data/run_config.json` e riletto da `01_baseline.ipynb` e dal fine-tuning.

Sul campione da 150 righe: segmento più lungo 12.550 token (Qwen3) → **16384**. Context nativo dei modelli: Qwen3 32k, SmolLM3 64k, Gemma 3 128k, quindi 16384 è supportato da tutti.

In [ ]:
import json
from slm_baseline.inference import recommend_max_seq_len

MAX_SEQ_LEN = recommend_max_seq_len(lengths, headroom=HEADROOM)
longest = {n.split("/")[-1]: int(l.max()) for n, l in lengths.items()}
print(f"segmento più lungo per tokenizer: {longest}")
print(f"max_seq_length raccomandato (+{HEADROOM:.0%}, multiplo di 1024): {MAX_SEQ_LEN}")

# I segmenti più lunghi: un singolo outlier può decidere la memoria GPU del training
ref = lengths[TOKENIZERS[0]]
display(sample.assign(seq_tokens=ref).nlargest(10, "seq_tokens")[["record_key", "macro_category", "sub_category", "seq_tokens"]])

RUN_CONFIG_PATH.write_text(json.dumps({
    "max_seq_len": MAX_SEQ_LEN, "headroom": HEADROOM, "longest_tokens": longest,
    "text_column": TEXT_COLUMN, "splits_path": str(SPLITS_PATH.relative_to(ROOT)),
}, indent=2))
print(f"scritto {RUN_CONFIG_PATH}")